# DEE — Análisis Edwards-Anderson: ¿el sustrato es vidrio?

**Pregunta:** ¿el sustrato DEE en su fase fundamental (T<θ_D) está en una fase de vidrio (no ergódica con múltiples mínimos metaestables) o en una fase ordenada (cristal con un único mínimo)?

**Método:** generar 6 pares de réplicas independientes (12 corridas MC totales), todas con la misma configuración inicial pero diferentes secuencias aleatorias de movimientos. Medir overlap entre réplicas y construir P(q).

**Predicciones:**
- Si **vidrio**: P(q) con múltiples picos (replica symmetry breaking), q_EA > 0 no uniforme, autocorrelación stretched exponential
- Si **cristal ordenado**: P(q) con un solo pico cerca de 1, q_EA > 0 uniforme
- Si **líquido**: P(q) con un solo pico cerca de 0, q_EA ≈ 0

**Configuración:**
- N = 500 (mismo que test espectral, comparación directa)
- 6 pares = 12 corridas MC, 25500 pasos cada una
- Topología: T³ (la que en SIM 18 mostró mejor convergencia)
- F = N·var(κ) + α·Σ(1−I)² (mismo de SIM 18)
- Tiempo estimado: ~3.5 horas

**Robustez:** guarda cada réplica a disco después de calcularla, así se puede retomar si la sesión se corta.

---

In [ ]:
import os
import numpy as np
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix, diags
from scipy.linalg import eigh
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import time
import json

PLOT_DIR = 'plots_glass'
DATA_DIR = 'data_glass'
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

def save_plot(name):
    plt.savefig(os.path.join(PLOT_DIR, f'{name}.png'), dpi=120, bbox_inches='tight')
    print(f'  -> {name}.png')

# Configuracion
THETA_D = 43.4
N_TARGET = 500
K = 20
L_T3 = 1.0
N_REPLICAS = 12  # 6 pares

print(f'N = {N_TARGET}, k = {K}')
print(f'Replicas = {N_REPLICAS} ({N_REPLICAS//2} pares)')
print(f'L (T³) = {L_T3}')

In [ ]:
# Funciones core (idénticas al test espectral)

def init_uniform_T3(N_target, seed=42, L=1.0):
    rng = np.random.default_rng(seed)
    return rng.uniform(0, L, size=(N_target, 3)), N_target

def neighbors_T3_fast(points, k=20, L=1.0):
    N = len(points)
    images = []
    image_to_orig = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                shift = np.array([dx*L, dy*L, dz*L])
                images.append(points + shift)
                image_to_orig.append(np.arange(N))
    images_all = np.concatenate(images, axis=0)
    image_to_orig_all = np.concatenate(image_to_orig)
    tree = cKDTree(images_all)
    
    nbrs = np.zeros((N, k), dtype=int)
    nbr_dists = np.zeros((N, k))
    nbr_pos = np.zeros((N, k, 3))
    for i in range(N):
        dists, idxs = tree.query(points[i], k=k+1)
        valid = ~((image_to_orig_all[idxs] == i) & (dists < 1e-12))
        v_idx = np.where(valid)[0][:k]
        nbrs[i] = image_to_orig_all[idxs[v_idx]]
        nbr_dists[i] = dists[v_idx]
        nbr_pos[i] = images_all[idxs[v_idx]]
    return nbrs, nbr_dists, nbr_pos

def laplacian_from_neighbors(nbrs, nbr_dists, alpha=2.0):
    N = len(nbrs)
    k = nbrs.shape[1]
    rows = np.repeat(np.arange(N), k)
    cols = nbrs.ravel()
    weights = (1.0 / nbr_dists**alpha).ravel()
    W = csr_matrix((weights, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    deg = np.array(W.sum(axis=1)).flatten()
    return diags(deg) - W, deg

def compute_F_vec(points, alpha_iso, alpha_kernel=2.0, k=K, L=L_T3):
    nbrs, nbr_dists, nbr_pos = neighbors_T3_fast(points, k=k, L=L)
    deg = np.sum(1.0 / nbr_dists**alpha_kernel, axis=1)
    kappa = deg / np.mean(deg)
    F_var = len(points) * np.var(kappa)
    
    if alpha_iso > 0:
        diffs = nbr_pos - points[:, None, :]
        M = np.einsum('ikj,ikm->ijm', diffs, diffs) / nbrs.shape[1]
        eigvals_all = np.linalg.eigvalsh(M)
        eigvals_all = np.maximum(eigvals_all, 1e-12)
        eigvals_top = eigvals_all[:, -3:] if eigvals_all.shape[1] > 3 else eigvals_all
        geom = np.exp(np.mean(np.log(eigvals_top), axis=1))
        arith = np.mean(eigvals_top, axis=1)
        iso = geom / arith
        F_iso = np.sum((1.0 - iso)**2)
        F_total = F_var + alpha_iso * F_iso
    else:
        iso = None
        F_total = F_var
        F_iso = 0
    return F_total, F_var, F_iso, kappa, iso

print('Funciones core listas')

In [ ]:
# Calibracion de alpha_iso (mismo procedimiento que SIM 18)
pts_init_for_calib, _ = init_uniform_T3(N_TARGET, seed=42, L=L_T3)
_, F_var_r, F_iso_r, _, _ = compute_F_vec(pts_init_for_calib, alpha_iso=1.0)
ALPHA_ISO = F_var_r / max(F_iso_r, 1e-6)
print(f'α_iso calibrado = {ALPHA_ISO:.4f}')
print(f'(igualdad de contribuciones para configuracion aleatoria)')

In [ ]:
# MC con annealing y guardado intermedio

def gen_move_T3(rng, sigma):
    return rng.normal(0, sigma, size=3)

def move_T3(points, i, delta, L=1.0):
    new_pts = points.copy()
    new_pts[i] = (points[i] + delta) % L
    return new_pts

def run_mc_replica(pts_init, T_schedule, n_steps_per_T, alpha_iso, sigma_move=0.02,
                    seed=42, log_every=2000):
    """
    Run MC annealing y retorna config final + historia + autocorrelacion.
    Guarda muestras de campo kappa periodicamente para calcular C(t).
    """
    rng = np.random.default_rng(seed)
    points = pts_init.copy()
    F_curr, _, _, _, _ = compute_F_vec(points, alpha_iso)
    
    history = {'step': [], 'T': [], 'F': [], 'kappa_var': [], 'iso_mean': []}
    kappa_samples = []  # snapshots de kappa para autocorrelacion
    sample_steps = []
    total_step = 0
    t_start = time.time()
    
    for T in T_schedule:
        n_acc = 0
        for s in range(n_steps_per_T):
            i = rng.integers(len(points))
            delta = gen_move_T3(rng, sigma_move)
            pts_new = move_T3(points, i, delta, L=L_T3)
            F_new, _, _, _, _ = compute_F_vec(pts_new, alpha_iso)
            dF = F_new - F_curr
            
            if T > 0:
                accept = (dF < 0) or (rng.random() < np.exp(-dF / T))
            else:
                accept = (dF < 0)
            
            if accept:
                points = pts_new
                F_curr = F_new
                n_acc += 1
            
            if total_step % log_every == 0:
                _, _, _, k_, iso_ = compute_F_vec(points, max(alpha_iso, 1.0))
                history['step'].append(total_step)
                history['T'].append(T)
                history['F'].append(F_curr)
                history['kappa_var'].append(k_.var())
                history['iso_mean'].append(iso_.mean() if iso_ is not None else 0)
                kappa_samples.append(k_.copy())
                sample_steps.append(total_step)
            total_step += 1
        
        ar = n_acc / n_steps_per_T
        elapsed = time.time() - t_start
        print(f'    T={T:6.2f}  acept={ar*100:5.1f}%  F={F_curr:.2f}  t={elapsed:.0f}s')
    
    # Final state
    _, _, _, kappa_final, iso_final = compute_F_vec(points, max(alpha_iso, 1.0))
    
    return {
        'pts_final': points,
        'kappa_final': kappa_final,
        'iso_final': iso_final,
        'F_final': F_curr,
        'history': history,
        'kappa_samples': np.array(kappa_samples),
        'sample_steps': np.array(sample_steps),
        'total_time': time.time() - t_start
    }

T_schedule_full = [THETA_D, THETA_D/3, THETA_D/10, 0.0]
n_steps_each = [7500, 7500, 4500, 6000]  # total 25500
print(f'Schedule: {T_schedule_full}')
print(f'Pasos por fase: {n_steps_each}, total {sum(n_steps_each)}')

In [ ]:
# CONFIGURACION INICIAL COMUN
# Todas las replicas parten de la MISMA config inicial,
# pero usan distinta seed para la secuencia de movimientos MC.

INITIAL_SEED = 42
pts_initial_common, _ = init_uniform_T3(N_TARGET, seed=INITIAL_SEED, L=L_T3)
F_init_common, _, _, kappa_init, _ = compute_F_vec(pts_initial_common, ALPHA_ISO)
print(f'Configuración inicial común (seed={INITIAL_SEED}):')
print(f'  N = {len(pts_initial_common)}')
print(f'  F inicial = {F_init_common:.2f}')
print(f'  var(κ) inicial = {kappa_init.var():.4f}')

np.save(os.path.join(DATA_DIR, 'pts_initial.npy'), pts_initial_common)
print(f'Configuración inicial guardada en {DATA_DIR}/pts_initial.npy')

---
## EJECUCIÓN AUTÓNOMA — 12 réplicas (6 pares)

Cada réplica se guarda a disco al terminar. Si la celda se corta a la mitad, al volver a ejecutar saltea las que ya están guardadas y continúa donde se quedó.

In [ ]:
# Loop principal: ejecutar las 12 replicas con distinta seed cada una
# Las seeds estan diseñadas para que pares (0,1), (2,3), (4,5), (6,7), (8,9), (10,11)
# sean replicas comparables

REPLICA_SEEDS = [101, 102, 201, 202, 301, 302, 401, 402, 501, 502, 601, 602]

all_replicas = {}
t_global_start = time.time()

for rep_idx, rep_seed in enumerate(REPLICA_SEEDS):
    fname = os.path.join(DATA_DIR, f'replica_{rep_idx:02d}_seed{rep_seed}.npz')
    
    # Skip si ya existe (reanudacion)
    if os.path.exists(fname):
        print(f'\n[{rep_idx+1}/{N_REPLICAS}] Réplica {rep_idx} (seed {rep_seed}) ya existe, cargando...')
        data = np.load(fname, allow_pickle=True)
        all_replicas[rep_idx] = {
            'pts_final': data['pts_final'],
            'kappa_final': data['kappa_final'],
            'iso_final': data['iso_final'],
            'F_final': float(data['F_final']),
            'kappa_samples': data['kappa_samples'],
            'sample_steps': data['sample_steps'],
            'seed': rep_seed,
            'time': float(data['total_time'])
        }
        print(f'    cargada (F_final = {all_replicas[rep_idx]["F_final"]:.2f})')
        continue
    
    print(f'\n{"="*70}')
    print(f'[{rep_idx+1}/{N_REPLICAS}] Réplica {rep_idx} (seed {rep_seed})')
    print(f'{"="*70}')
    elapsed_global = time.time() - t_global_start
    if rep_idx > 0:
        avg_per_replica = elapsed_global / rep_idx
        eta_min = avg_per_replica * (N_REPLICAS - rep_idx) / 60
        print(f'Tiempo transcurrido: {elapsed_global/60:.0f} min, ETA total: {eta_min:.0f} min restantes')
    
    # Annealing por fases
    pts_curr = pts_initial_common.copy()
    full_history = {'step': [], 'T': [], 'F': [], 'kappa_var': [], 'iso_mean': []}
    all_kappa_samples = []
    all_sample_steps = []
    offset = 0
    t_replica_start = time.time()
    F_curr = F_init_common
    
    for T_phase, n_steps in zip(T_schedule_full, n_steps_each):
        print(f'\n  Fase T = {T_phase:.2f}, {n_steps} pasos:')
        result = run_mc_replica(
            pts_curr, [T_phase], n_steps,
            alpha_iso=ALPHA_ISO, sigma_move=0.02,
            seed=rep_seed + offset
        )
        pts_curr = result['pts_final']
        F_curr = result['F_final']
        for kk in result['history']:
            if kk == 'step':
                full_history[kk].extend([s + offset for s in result['history'][kk]])
            else:
                full_history[kk].extend(result['history'][kk])
        all_kappa_samples.append(result['kappa_samples'])
        all_sample_steps.append(result['sample_steps'] + offset)
        offset += n_steps
    
    kappa_samples_total = np.concatenate(all_kappa_samples, axis=0)
    sample_steps_total = np.concatenate(all_sample_steps)
    
    # Mediciones finales
    _, _, _, kappa_final, iso_final = compute_F_vec(pts_curr, max(ALPHA_ISO, 1.0))
    
    elapsed_replica = time.time() - t_replica_start
    print(f'\n  ✓ Réplica {rep_idx} completada en {elapsed_replica:.0f}s ({elapsed_replica/60:.1f} min)')
    print(f'    F final = {F_curr:.2f}, var(κ) = {kappa_final.var():.5f}, ⟨I⟩ = {iso_final.mean():.4f}')
    
    # Guardar
    np.savez(fname,
             pts_final=pts_curr,
             kappa_final=kappa_final,
             iso_final=iso_final,
             F_final=F_curr,
             kappa_samples=kappa_samples_total,
             sample_steps=sample_steps_total,
             total_time=elapsed_replica)
    print(f'    guardada en {fname}')
    
    all_replicas[rep_idx] = {
        'pts_final': pts_curr,
        'kappa_final': kappa_final,
        'iso_final': iso_final,
        'F_final': F_curr,
        'kappa_samples': kappa_samples_total,
        'sample_steps': sample_steps_total,
        'seed': rep_seed,
        'time': elapsed_replica
    }

elapsed_total = time.time() - t_global_start
print(f'\n{"="*70}')
print(f'TODAS LAS RÉPLICAS COMPLETADAS')
print(f'Tiempo total: {elapsed_total/60:.1f} min ({elapsed_total/3600:.2f} h)')
print(f'{"="*70}')

---
## ANÁLISIS — Edwards-Anderson y P(q)

In [ ]:
# Calcular metricas finales por replica
print('Resumen por réplica:\n')
print(f'{"Rep":>4} {"seed":>5} {"F_final":>10} {"var(κ)":>10} {"⟨I⟩":>8}')
print('-'*50)
for ridx in sorted(all_replicas.keys()):
    r = all_replicas[ridx]
    print(f'{ridx:>4} {r["seed"]:>5} {r["F_final"]:>10.2f} '
          f'{r["kappa_final"].var():>10.5f} {r["iso_final"].mean():>8.4f}')

In [ ]:
# OVERLAP ENTRE REPLICAS — la metrica clave de Edwards-Anderson
# 
# Como los nodos comparten el mismo indice (todas parten de pts_initial_common),
# podemos calcular q_12 nodo a nodo:
# q_12 = (1/N) * sum_i kappa_1(i) * kappa_2(i) / (var_1 * var_2)^(1/2)
# 
# Pero las posiciones espaciales pueden haber permutado, asi que tambien
# calculamos overlap basado en distribuciones invariantes a permutacion.

def overlap_kappa(rep_a, rep_b):
    """Overlap nodo-a-nodo en kappa centrada y normalizada."""
    k1 = rep_a['kappa_final'] - 1.0  # centrada (kappa esta normalizada a 1)
    k2 = rep_b['kappa_final'] - 1.0
    norm = np.sqrt(np.var(k1) * np.var(k2))
    if norm < 1e-12:
        return 1.0  # ambos uniformes
    return np.mean(k1 * k2) / norm

def overlap_position(rep_a, rep_b, L=L_T3):
    """Overlap nodo-a-nodo en posiciones (centradas)."""
    pa = rep_a['pts_final']
    pb = rep_b['pts_final']
    diff = pa - pb
    diff = diff - L * np.round(diff / L)  # distancia periodica
    dist_per_node = np.linalg.norm(diff, axis=1)
    # Si overlap positivo, distancias chicas; si overlap=0, distancias ~ L/2
    mean_dist = dist_per_node.mean()
    return 1.0 - 2 * mean_dist / L  # rango [-1, 1]

def overlap_distribution(rep_a, rep_b):
    """Overlap entre distribuciones de kappa (invariante permutacion)."""
    k1_sorted = np.sort(rep_a['kappa_final'])
    k2_sorted = np.sort(rep_b['kappa_final'])
    diff = k1_sorted - k2_sorted
    return 1.0 - np.std(diff) / np.std(k1_sorted)

# Calcular todos los overlaps entre pares
print('Overlap entre todos los pares de replicas:\n')
print(f'{"i,j":>6} {"q_kappa":>10} {"q_pos":>10} {"q_dist":>10}')
print('-'*45)

all_q_kappa = []
all_q_pos = []
all_q_dist = []
pair_indices = []

for i in range(N_REPLICAS):
    for j in range(i+1, N_REPLICAS):
        q_k = overlap_kappa(all_replicas[i], all_replicas[j])
        q_p = overlap_position(all_replicas[i], all_replicas[j])
        q_d = overlap_distribution(all_replicas[i], all_replicas[j])
        all_q_kappa.append(q_k)
        all_q_pos.append(q_p)
        all_q_dist.append(q_d)
        pair_indices.append((i, j))
        print(f'{i:>2},{j:<2}  {q_k:>10.4f} {q_p:>10.4f} {q_d:>10.4f}')

all_q_kappa = np.array(all_q_kappa)
all_q_pos = np.array(all_q_pos)
all_q_dist = np.array(all_q_dist)

print(f'\n{len(all_q_kappa)} pares calculados (= C({N_REPLICAS},2))')

In [ ]:
# DISTRIBUCION P(q) — la firma de fase de vidrio
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, q_vals, name, color in [
    (axes[0], all_q_kappa, 'q (overlap κ nodo-a-nodo)', 'crimson'),
    (axes[1], all_q_pos, 'q (overlap posición)', 'steelblue'),
    (axes[2], all_q_dist, 'q (overlap distribución κ)', 'darkgreen'),
]:
    ax.hist(q_vals, bins=20, color=color, alpha=0.75, edgecolor='black')
    ax.axvline(q_vals.mean(), color='black', linestyle='--', lw=2,
               label=f'⟨q⟩ = {q_vals.mean():.3f}')
    ax.axvline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xlabel(name, fontsize=11)
    ax.set_ylabel('Conteo', fontsize=11)
    ax.set_title(f'P(q): {name.split("(")[0]}', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle('P(q) — Distribución de overlaps entre réplicas\n'
             '(múltiples picos = vidrio; un pico = cristal o líquido)',
             fontsize=12, y=1.02)
plt.tight_layout()
save_plot('150_distribucion_q')
plt.show()

In [ ]:
# Diagnostico: ¿es vidrio?
print('='*72)
print('DIAGNÓSTICO DE FASE DE VIDRIO')
print('='*72)
print()

for q_vals, name in [(all_q_kappa, 'overlap κ nodo-a-nodo'),
                      (all_q_pos, 'overlap posición'),
                      (all_q_dist, 'overlap distribución')]:
    mean_q = q_vals.mean()
    std_q = q_vals.std()
    range_q = q_vals.max() - q_vals.min()
    
    print(f'\n{name}:')
    print(f'  ⟨q⟩ = {mean_q:.4f} ± {std_q:.4f}')
    print(f'  rango = [{q_vals.min():.4f}, {q_vals.max():.4f}], ancho = {range_q:.4f}')
    
    # Tests heuristicos
    if std_q > 0.15 or range_q > 0.4:
        print(f'  ✓ Distribucion ANCHA (std/range altos): consistente con VIDRIO')
    elif std_q < 0.05 and abs(mean_q) < 0.1:
        print(f'  → Distribucion estrecha cerca de q=0: LÍQUIDO o ergodico')
    elif std_q < 0.05 and mean_q > 0.5:
        print(f'  → Distribucion estrecha cerca de q alto: CRISTAL ordenado')
    else:
        print(f'  ~ Distribucion intermedia')

# Test de bimodalidad
from scipy.stats import skew, kurtosis
for q_vals, name in [(all_q_kappa, 'kappa'), (all_q_pos, 'pos'), (all_q_dist, 'dist')]:
    sk = skew(q_vals)
    kt = kurtosis(q_vals)
    print(f'\nMomentos de q ({name}): skew = {sk:.3f}, kurtosis = {kt:.3f}')
    if kt < -0.5:
        print(f'   → Kurtosis negativa: distribucion BIMODAL/PLANA, sugiere vidrio')
    elif kt > 1:
        print(f'   → Kurtosis alta: pico unico definido')

In [ ]:
# AUTOCORRELACION TEMPORAL — firma dinamica de vidrio
# C(t) = (1/N) sum_i kappa_i(t0) * kappa_i(t0 + t) / var
# En vidrio: stretched exponential C(t) ~ exp(-(t/tau)^beta), beta < 1
# En liquido: exponencial puro, beta = 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for ridx in range(min(6, N_REPLICAS)):  # primeras 6 replicas
    r = all_replicas[ridx]
    samples = r['kappa_samples']
    if len(samples) < 3:
        continue
    
    # C(t) = correlacion entre snapshot inicial y siguientes
    k_t0 = samples[0] - 1.0
    var_t0 = np.var(k_t0)
    if var_t0 < 1e-10:
        continue
    
    C_t = []
    for t_idx in range(len(samples)):
        k_t = samples[t_idx] - 1.0
        c = np.mean(k_t0 * k_t) / var_t0
        C_t.append(c)
    
    ax.plot(r['sample_steps'], C_t, '-', alpha=0.7, label=f'Rep {ridx}')

ax.set_xlabel('Paso MC', fontsize=11)
ax.set_ylabel('C(t) (autocorrelación normalizada)', fontsize=11)
ax.set_title('Autocorrelación temporal de κ\n(stretched exp = vidrio; exp puro = liquido)', fontsize=11)
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
ax.set_yscale('symlog', linthresh=0.01)

# Promedio de C(t)
ax = axes[1]
all_C = []
min_len = min(len(all_replicas[i]['kappa_samples']) for i in range(N_REPLICAS))
common_steps = all_replicas[0]['sample_steps'][:min_len]
for ridx in range(N_REPLICAS):
    samples = all_replicas[ridx]['kappa_samples'][:min_len]
    k_t0 = samples[0] - 1.0
    var_t0 = np.var(k_t0)
    if var_t0 < 1e-10:
        continue
    C_t = [np.mean((samples[t]-1.0) * k_t0) / var_t0 for t in range(min_len)]
    all_C.append(C_t)

if all_C:
    all_C = np.array(all_C)
    C_mean = all_C.mean(axis=0)
    C_std = all_C.std(axis=0)
    
    ax.errorbar(common_steps, C_mean, yerr=C_std, fmt='o-',
                color='crimson', lw=2, markersize=6, capsize=4)
    ax.set_xlabel('Paso MC', fontsize=11)
    ax.set_ylabel('⟨C(t)⟩', fontsize=11)
    ax.set_title('Autocorrelación promediada', fontsize=11)
    ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('151_autocorrelacion')
plt.show()

In [ ]:
# Visualizacion final: F_final por replica + distribucion de var(kappa)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F final
ax = axes[0]
F_finales = [all_replicas[i]['F_final'] for i in range(N_REPLICAS)]
var_finales = [all_replicas[i]['kappa_final'].var() for i in range(N_REPLICAS)]
iso_finales = [all_replicas[i]['iso_final'].mean() for i in range(N_REPLICAS)]

ax.bar(range(N_REPLICAS), F_finales, color='crimson', alpha=0.7, edgecolor='black')
ax.axhline(np.mean(F_finales), color='black', linestyle='--',
           label=f'⟨F⟩ = {np.mean(F_finales):.2f} ± {np.std(F_finales):.2f}')
ax.set_xlabel('Réplica', fontsize=11)
ax.set_ylabel('F final', fontsize=11)
ax.set_title('F final por réplica\n(dispersión = paisaje rugoso)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='y')

# Histograma de F vs Var(F): coeficiente de variacion
ax = axes[1]
F_arr = np.array(F_finales)
cv_F = F_arr.std() / F_arr.mean()
ax.hist(F_finales, bins=8, color='steelblue', alpha=0.7, edgecolor='black')
ax.axvline(F_arr.mean(), color='red', linestyle='--', lw=2,
           label=f'⟨F⟩ = {F_arr.mean():.2f}')
ax.set_xlabel('F final', fontsize=11)
ax.set_ylabel('Conteo', fontsize=11)
ax.set_title(f'Distribución de F finales\nCV = σ/⟨F⟩ = {cv_F:.3f}', fontsize=11)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('152_F_finales')
plt.show()

print(f'\nCoeficiente de variación de F finales: {cv_F:.3f}')
print(f'  Si CV > 0.1: paisaje rugoso (replicas terminan en distintos minimos)')
print(f'  Si CV < 0.05: replicas convergen al mismo minimo (cristal limpio)')

In [ ]:
# VEREDICTO FINAL
print('='*72)
print('VEREDICTO — ¿el sustrato DEE está en fase de vidrio?')
print('='*72)
print()

# Criterios:
# 1. P(q) ancho con std > 0.15
# 2. CV(F) > 0.1
# 3. Kurtosis negativa

criteria_met = 0
criteria_total = 4

# Criterio 1: ancho de P(q_kappa)
if all_q_kappa.std() > 0.15:
    print(f'✓ Criterio 1: P(q_κ) ancho (std={all_q_kappa.std():.3f} > 0.15)')
    criteria_met += 1
else:
    print(f'✗ Criterio 1: P(q_κ) estrecho (std={all_q_kappa.std():.3f} < 0.15)')

# Criterio 2: CV de F
if cv_F > 0.1:
    print(f'✓ Criterio 2: F finales dispersos (CV={cv_F:.3f} > 0.1)')
    criteria_met += 1
else:
    print(f'✗ Criterio 2: F finales convergentes (CV={cv_F:.3f} < 0.1)')

# Criterio 3: kurtosis de q
kt_q = kurtosis(all_q_kappa)
if kt_q < -0.3:
    print(f'✓ Criterio 3: P(q) bimodal/plana (kurtosis={kt_q:.3f})')
    criteria_met += 1
else:
    print(f'✗ Criterio 3: P(q) unimodal (kurtosis={kt_q:.3f})')

# Criterio 4: rango de q
if all_q_kappa.max() - all_q_kappa.min() > 0.4:
    print(f'✓ Criterio 4: Rango amplio de q ({all_q_kappa.max()-all_q_kappa.min():.3f})')
    criteria_met += 1
else:
    print(f'✗ Criterio 4: Rango estrecho de q ({all_q_kappa.max()-all_q_kappa.min():.3f})')

print(f'\nCriterios pasados: {criteria_met}/{criteria_total}')
print()
if criteria_met >= 3:
    print('VEREDICTO: VIDRIO (replica symmetry breaking detectado)')
    print('  El sustrato DEE en fase fundamental tiene multiples minimos metaestables')
    print('  no equivalentes. Esto valida la interpretacion glass-like sugerida por SIM 18')
    print('  y provee marco termodinamico para los defectos blandos de SIM 13.')
elif criteria_met == 2:
    print('VEREDICTO: VIDRIO MARGINAL (transicion de vidrio en proximidades)')
    print('  El sustrato vive cerca de la transicion pero no claramente dentro de ella.')
    print('  Comportamiento intermedio entre cristal y vidrio.')
elif criteria_met == 1:
    print('VEREDICTO: CRISTAL CON DEFECTOS (no es vidrio claro)')
    print('  El sustrato tiene un mínimo dominante pero con defectos puntuales.')
else:
    print('VEREDICTO: ERGÓDICO (no hay rotura de simetria de replica)')
    print('  El sustrato converge al mismo mínimo desde distintas evoluciones.')
    print('  No hay paisaje rugoso significativo.')

In [ ]:
# Empaquetar resultados para descarga
import shutil
shutil.make_archive('plots_glass', 'zip', PLOT_DIR)
shutil.make_archive('data_glass', 'zip', DATA_DIR)

try:
    from google.colab import files
    files.download('plots_glass.zip')
    files.download('data_glass.zip')
except ImportError:
    pass

print('Listo.')
print('Plots en plots_glass.zip')
print('Datos crudos en data_glass.zip')
print(f'Tiempo total: {(time.time() - t_global_start)/60:.1f} min')